In [2]:
import torch
from torchvision.models import resnet50, ResNet50_Weights
import os
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import json
import random
import time

In [3]:
# Default model set for speed: 'yolov5s'. 
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
# Load yolov5 model with pytorch
def load_yolo_model(model_type):
    # Load the YOLO model
    model = torch.hub.load('ultralytics/yolov5', model_type, pretrained=True)
    if torch.cuda.is_available():
        model = model.to('cuda')
    return model

Using cache found in /Users/robo/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2024-4-12 Python-3.11.8 torch-2.2.2 CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


In [4]:
def lookup_object_index(object_name, model):
    # Use class names set in model.
    class_names = model.names
    # Create a dictionary to map class names to indices.
    name_to_index = {name: idx for idx, name in class_names.items()}
    if object_name not in name_to_index:
        return None
    return name_to_index[object_name]

In [5]:
def detect_classify_and_count(object_name, image_dir, output_dir, detection_model, threshold=0.6):
    # check if object_name is in the yolo model's class names
    object_index = lookup_object_index(object_name, detection_model)
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    # Initialize ResNet50
    weights = ResNet50_Weights.DEFAULT
    classification_model = resnet50(weights=weights)
    classification_model.eval()
    preprocess = weights.transforms()

    results_list = []
    start_time = time.time()
    total_images_with_objects = 0
    total_objects = 0
    # Loop through all images in the image_dir
    for image_path in os.listdir(image_dir):
        full_path = os.path.join(image_dir, image_path)
        if image_path.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                # verify the image
                with Image.open(full_path) as img:
                    img.verify()
                    img = Image.open(full_path)

                results = detection_model(img)
                predictions = results.xyxy[0]
                # Initialize the image draw object
                draw = ImageDraw.Draw(img)
                font = ImageFont.load_default()

                object_count = 1

                for *box, conf, cls_idx in predictions:
                    draw.rectangle(box, outline='blue', width=2)
                    draw.text((box[0], box[1]), f"{category_name}: {score:.2f}", fill='blue', font=font)
                    # If object_index matches search, then add bounding box and count
                    # Split the box region into crop values
                    x1, y1, x2, y2 = map(lambda x: round(x.item()), box)
                    # Crop the into regions
                    region = img.crop((x1, y1, x2, y2))
                    # Preprocess the region to claissify the object it contains.
                    region_tensor = preprocess(region).unsqueeze(0)
                    prediction = classification_model(region_tensor).squeeze(0).softmax(0)
                    class_id = prediction.argmax().item()
                    # Get the score and category name
                    score = prediction[class_id].item()
                    category_name = weights.meta["categories"][class_id]
                    # If score is greater than threshold, and input name matches category name
                    if (score > threshold) and (category_name == object_name):
                        object_count += 1
                        draw.rectangle(box, outline='green', width=2)
                        draw.text((box[0], box[1]), f"{object_name}: {score:.2f}", fill='green', font=font)
                    # if cls_idx == object_index and conf > threshold:
                        # object_count += 1
                        # draw.rectangle(box, outline='red', width=2)
                        # draw.text((box[0], box[1]), f"{object_name}: {conf:.2f}", fill='red', font=font)
                    # # If object_index is False, classify the region
                    # if (object_index is None) or (conf < threshold):
                    #     # Split the box region into crop values
                    #     x1, y1, x2, y2 = map(lambda x: round(x.item()), box)
                    #     # Crop the into regions
                    #     region = img.crop((x1, y1, x2, y2))
                    #     # Preprocess the region to claissify the object it contains.
                    #     region_tensor = preprocess(region).unsqueeze(0)
                    #     prediction = classification_model(region_tensor).squeeze(0).softmax(0)
                    #     class_id = prediction.argmax().item()
                    #     # Get the score and category name
                    #     score = prediction[class_id].item()
                    #     category_name = weights.meta["categories"][class_id]
                    #     # If score is greater than threshold, and input name matches category name
                    #     if (score > threshold) and (category_name == object_name):
                    #         object_count += 1
                    #         draw.rectangle(box, outline='green', width=2)
                    #         draw.text((box[0], box[1]), f"{object_name}: {score:.2f}", fill='green', font=font)
                    #     # If score is less than threshold then box is add for reference.
                    #     if score < threshold:
                    #         draw.rectangle(box, outline='blue', width=2)
                    #         draw.text((box[0], box[1]), f"{category_name}: {score:.2f}", fill='blue', font=font)
                # If object_count is greater than 0, save the image
                if object_count > 0:
                    total_images_with_objects += 1
                    total_objects = total_objects + object_count
                    save_path = os.path.join(output_dir, image_path)
                    img.save(save_path)
                    results_list.append({'image': image_path, 'object_count': object_count, 'saved_to': save_path})

            except Exception as e:
                print(f"Error processing {full_path}: {e}")

    total_time = time.time() - start_time
    results_summary = {
        'total_images_with_objects': total_images_with_objects,
        'total_objects': total_objects,
        'total_processing_time': total_time,
        'results': results_list
    }
    
    with open(os.path.join(output_dir, 'results.json'), 'w') as f:
        json.dump(results_summary, f, indent=4)

    return json.dumps(results_summary, indent=4)

In [6]:
# Example usage
object_name = 'apple'
# Set the threshold for detection confidence
threshold = 0.4
# Target of imaages
image_dir = 'Datasets/FSC147_384_V2/images_384_VarV2'
# Output directory
output_dir = 'Results/'+object_name+'_results_'+str(random.randint(1, 100))

# You can change it to 'yolov5s', 'yolov5m', 'yolov5l', 'yolov5x
modelType = 'yolov5x'
# Load the YOLO model
model = load_yolo_model(modelType)
# JSON results
results_json = detect_classify_and_count(object_name, image_dir, output_dir, model, threshold)
print(results_json)

Using cache found in /Users/robo/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2024-4-12 Python-3.11.8 torch-2.2.2 CPU

Fusing layers... 
YOLOv5x summary: 444 layers, 86705005 parameters, 0 gradients, 205.5 GFLOPs
Adding AutoShape... 


KeyboardInterrupt: 

The below is just for an example of a real world implementation.

In [ ]:
# Potential way to run the code as a Flask API
from flask import Flask, request, jsonify
app = Flask(__name__)

@app.route('/set_variables', methods=['GET'])
def set_variables():
    object_name = request.args.get('object_name', 'car')  # Default value is 'car'
    image_dir = request.args.get('image_dir', 'Datasets/FSC147_384_V2/images_384_VarV2')  # Default value
    output_dir = 'Results/' + object_name + '_results' + str(random.randint(1, 100))
    modelType = request.args.get('modelType', 'yolov5x')  # Default value is 'yolov5x'
    threshold = float(request.args.get('threshold', 0.7))  # Default value is 0.7

    # You can change it to 'yolov5s', 'yolov5m', 'yolov5l', 'yolov5x'
    model = load_yolo_model(modelType)
    results_json = detect_and_count(object_name, image_dir, output_dir, model, threshold)
    
    return jsonify(results_json)

if __name__ == '__main__':
    app.run(debug=True)